# Week 36, Day 1: Population vectors and Bayesian decoding

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w036/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Population vector against maximum likelihood

Complete pop_vector(n, pd), returning the angle of the sum of (n - baseline counts) times each neuron's unit preferred-direction vector. The maximum-likelihood decoder, which searches a grid of directions for the one that makes the counts most probable under Poisson tuning curves, is given.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(1986)
N, T, base = 60, 0.5, 5.0
grid = np.linspace(0, 2 * np.pi, 360, endpoint=False)
pds = {"uniform": r.uniform(0, 2 * np.pi, N),
       "clustered": np.r_[r.uniform(0, np.pi / 2, 40), r.uniform(0, 2 * np.pi, 20)]}
rates = lambda th, pd: base + 15 * np.maximum(0, np.cos(th - pd))

def pop_vector(n, pd):
    v = ...
    return np.arctan2(v[1], v[0])

def ml_decode(n, pd):
    F = rates(grid[:, None], pd) * T
    return grid[np.argmax(np.log(F) @ n - F.sum(1))]

angdiff = lambda a, b: np.degrees(np.abs(np.angle(np.exp(1j * (a - b)))))
err = {}
for name, pd in pds.items():
    e = {"PV": [], "ML": []}
    for _ in range(1000):
        th = r.uniform(0, 2 * np.pi); n = r.poisson(rates(th, pd) * T)
        e["PV"].append(angdiff(pop_vector(n, pd), th)); e["ML"].append(angdiff(ml_decode(n, pd), th))
    err[name] = {k: float(np.mean(v)) for k, v in e.items()}
print({k: {m: round(x, 1) for m, x in v.items()} for k, v in err.items()}, "mean error in degrees")

In [ ]:
import numpy as np
def test_uniform():
    "with uniform preferred directions, the population vector errs by less than 15 degrees on average"
    assert err["uniform"]["PV"] < 15

def test_biased():
    "with clustered preferred directions, the population vector error at least doubles, while maximum likelihood stays under 6 degrees"
    assert err["clustered"]["PV"] > 2 * err["uniform"]["PV"] and err["clustered"]["ML"] < 6

test_uniform()
test_biased()
print("All checks passed")

### Exercise: Place fields and the posterior

Write log_posterior(n, win), returning the log posterior over the 50 position bins (flat prior) for a vector of counts n in a window of win seconds, using the estimated place fields tc (50 bins by 20 cells). Then decode the held-out half in 250 ms windows.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(1998)
L, dt, dur = 100, 0.01, 1200
pos = np.zeros(int(dur / dt)); x, v = 50.0, 20.0
for i in range(len(pos)):
    v += -0.5 * (v - np.sign(v) * 25) * dt + 8 * np.sqrt(dt) * r.normal()
    x += v * dt
    if x > L: x, v = 2 * L - x, -v
    if x < 0: x, v = -x, -v
    pos[i] = x
NC = 20
centres = r.uniform(0, L, NC); width = r.uniform(5, 12, NC); peak = r.uniform(3, 12, NC)
spk = r.poisson((0.3 + peak * np.exp(-0.5 * ((pos[:, None] - centres) / width) ** 2)) * dt)

half = len(pos) // 2
bins = np.linspace(0, L, 51); bc = (bins[:-1] + bins[1:]) / 2
occ = np.histogram(pos[:half], bins)[0] * dt
tc = np.array([np.histogram(pos[:half], bins, weights=spk[:half, i])[0] for i in range(NC)]).T / occ[:, None]
tc = np.maximum(tc, 0.05)                      # avoid log(0) where a cell never fired

def windows(win):
    w = int(round(win / dt)); m = (len(pos) - half) // w
    n = spk[half:half + m * w].reshape(m, w, NC).sum(1)
    truth = pos[half:half + m * w].reshape(m, w).mean(1)
    return n, truth

def log_posterior(n, win):
    return ...

n, truth = windows(0.25)
est = np.array([bc[np.argmax(log_posterior(k, 0.25))] for k in n])
err_250 = np.median(np.abs(est - truth))
print(f"median error with 250 ms windows: {err_250:.1f} cm")
plt.plot(truth[:200], "k", label="true"); plt.plot(est[:200], ".", label="decoded"); plt.ylabel("position (cm)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_fields():
    "the estimated place fields peak within 6 cm of the true centres for most cells"
    peaks = bc[np.argmax(tc, 0)]
    assert np.mean(np.abs(peaks - centres) < 6) > 0.8

def test_decode():
    "with 250 ms windows, the median decoding error is below 5 cm"
    assert err_250 < 5

test_fields()
test_decode()
print("All checks passed")

### Exercise: Adding a continuity prior

Complete decode_filter: for each window, add the log of the spread previous posterior (K @ prev) to the log likelihood, normalise, and store the peak. The spread width is 1.5 times the distance the animal can travel at 25 cm/s in one window, with a minimum of 2 cm.

In [ ]:
def decode_filter(win, use_prior):
    n, truth = windows(win)
    sig = max(2.0, 1.5 * 25 * win)
    K = np.exp(-0.5 * ((bc[:, None] - bc[None, :]) / sig) ** 2)
    prev, est = None, []
    for k in n:
        lp = log_posterior(k, win)
        if use_prior and prev is not None:
            lp = ...
        p = np.exp(lp - lp.max()); p /= p.sum(); prev = p
        est.append(bc[np.argmax(p)])
    return np.median(np.abs(np.array(est) - truth))

errs = {w: {"flat": decode_filter(w, False), "prior": decode_filter(w, True)} for w in (0.05, 0.1, 0.25, 0.5, 1.0)}
for w, e in errs.items():
    print(f"{int(w * 1000):5d} ms: flat {e['flat']:.1f} cm, with prior {e['prior']:.1f} cm")

In [ ]:
import numpy as np
def test_short():
    "at 50 ms, the continuity prior reduces the median error by at least 25%"
    assert errs[0.05]["prior"] < 0.75 * errs[0.05]["flat"]

def test_tradeoff():
    "with a flat prior, the error at 50 ms is more than twice the error at 500 ms"
    assert errs[0.05]["flat"] > 2 * errs[0.5]["flat"]

test_short()
test_tradeoff()
print("All checks passed")